# Step 3 — LLM Inference Pipeline (Ollama)

Loads Davidson tweets (with Jigsaw-classifier predictions from Step 1/2), runs each tweet through
**llama3.2:latest** using three prompt strategies, then compares macro F1 against Thrisha's classifier.

| Strategy | Description |
|---|---|
| Zero-shot | Single instruction, no examples |
| Few-shot | Three labeled examples before the query |
| Chain-of-thought | Model reasons step-by-step before labeling |

**Prerequisites:** `ollama serve` running, model pulled (`ollama pull llama3.2:latest`)

In [ ]:
import os
import pandas as pd
import ollama
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score

NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

## Configuration

In [ ]:
INPUT_CSV         = os.path.join(RESULTS_DIR, 'davidson_predictions.csv')
OUTPUT_CSV        = os.path.join(RESULTS_DIR, 'llm_predictions.csv')
MODEL_NAME        = 'llama3.2:latest'
TEXT_COL          = 'text'
LABEL_COL         = 'is_toxic'          # ground truth: 0=non-toxic, 1=toxic
BASELINE_PRED_COL = 'prediction'        # Thrisha's Jigsaw-trained classifier

# Full dataset = 24,783 rows × 3 strategies ≈ hours of inference locally.
# Keep MAX_SAMPLES ≤ 500 for a reasonable runtime. Set to None for the full run.
MAX_SAMPLES = 200

## Load Data

In [ ]:
df = pd.read_csv(INPUT_CSV)
df[LABEL_COL]         = df[LABEL_COL].astype(int)
df[BASELINE_PRED_COL] = df[BASELINE_PRED_COL].astype(int)

if MAX_SAMPLES is not None:
    n_per_class = MAX_SAMPLES // 2
    df = pd.concat([
        grp.sample(n=min(len(grp), n_per_class), random_state=42)
        for _, grp in df.groupby(LABEL_COL)
    ]).reset_index(drop=True)

print(f'Using {len(df):,} rows')
print('Label distribution:')
print(df[LABEL_COL].value_counts().rename({0: 'non-toxic', 1: 'toxic'}))
df[[TEXT_COL, LABEL_COL, BASELINE_PRED_COL]].head()

## Prompt Strategies

All three strategies ask the model to use the original Davidson label space:
**hate speech** / **offensive** / **neither**.
The parse step maps those to binary (hate speech + offensive → 1, neither → 0)
so results are comparable to the binary ground truth and Thrisha's classifier.

In [ ]:
# ── Zero-shot ────────────────────────────────────────────────────────────────
SYSTEM_ZS = (
    'You are a content moderation assistant. '
    'Classify the tweet into exactly one of these three categories: '
    '"hate speech", "offensive", or "neither". '
    'Hate speech directly attacks people based on protected characteristics such as race, '
    'religion, gender, or sexual orientation. '
    'Offensive language is vulgar or hurtful but not targeted hate speech. '
    'Neither means the tweet is not hateful or offensive. '
    'Respond with ONLY one of: hate speech, offensive, neither. Do not explain.'
)

def prompt_zero_shot(tweet: str) -> list:
    return [
        {'role': 'system', 'content': SYSTEM_ZS},
        {'role': 'user',   'content': f'Tweet: "{tweet}"'},
    ]


# ── Few-shot ─────────────────────────────────────────────────────────────────
FEW_SHOT_EXAMPLES = [
    (
        'go back to your country you filthy [slur], you don't belong here',
        'hate speech',
    ),
    (
        'this bitch is so fucking annoying i can't stand her lmao',
        'offensive',
    ),
    (
        'just grabbed coffee with my coworker, great start to the morning',
        'neither',
    ),
]

SYSTEM_FS = (
    'You are a content moderation assistant. '
    'Classify tweets as "hate speech", "offensive", or "neither". '
    'Respond with ONLY one of those three labels.'
)

def prompt_few_shot(tweet: str) -> list:
    messages = [{'role': 'system', 'content': SYSTEM_FS}]
    for ex_tweet, ex_label in FEW_SHOT_EXAMPLES:
        messages.append({'role': 'user',      'content': f'Tweet: "{ex_tweet}"'})
        messages.append({'role': 'assistant', 'content': ex_label})
    messages.append({'role': 'user', 'content': f'Tweet: "{tweet}"'})
    return messages


# ── Chain-of-thought ─────────────────────────────────────────────────────────
SYSTEM_COT = (
    'You are a content moderation assistant. '
    'Classify tweets as "hate speech", "offensive", or "neither". '
    'First, briefly reason about the content in 1-2 sentences. '
    'Then end your response with exactly this format on its own line: '
    'Label: <hate speech|offensive|neither>'
)

def prompt_cot(tweet: str) -> list:
    return [
        {'role': 'system', 'content': SYSTEM_COT},
        {'role': 'user',   'content': f'Tweet: "{tweet}"'},
    ]

## Label Parser

In [ ]:
def parse_label(raw: str) -> int:
    """
    Maps a model response to a binary label.
    For CoT responses, extracts the text after the last 'Label:' marker.
    Returns: 1 (toxic) for hate speech or offensive, 0 (non-toxic) for neither, -1 if unparseable.
    """
    text = raw.lower().strip()
    if 'label:' in text:
        text = text.split('label:')[-1].strip()
    if 'hate' in text:
        return 1
    if 'offensive' in text:
        return 1
    if 'neither' in text:
        return 0
    return -1

## Run Inference (all 3 strategies)

In [ ]:
STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot':  prompt_few_shot,
    'cot':       prompt_cot,
}

for strategy_name, prompt_fn in STRATEGIES.items():
    raw_col  = f'{strategy_name}_raw'
    pred_col = f'{strategy_name}_pred'
    raws, preds = [], []

    for tweet in tqdm(df[TEXT_COL].astype(str), desc=f'[{strategy_name}]'):
        try:
            response = ollama.chat(model=MODEL_NAME, messages=prompt_fn(tweet))
            raw = response['message']['content'].strip()
        except Exception as e:
            raw = f'ERROR: {e}'
        raws.append(raw)
        preds.append(parse_label(raw))

    df[raw_col]  = raws
    df[pred_col] = preds
    n_bad = preds.count(-1)
    print(f'{strategy_name}: {len(preds) - n_bad}/{len(preds)} parsed ({n_bad} unparseable)')

## Save Predictions

In [ ]:
df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(df):,} rows → {OUTPUT_CSV}')

## Evaluation — F1 Comparison

Rows where the LLM returned an unparseable response (`pred == -1`) are excluded
from that method's evaluation so scores remain comparable.

In [ ]:
METHODS = {
    "Thrisha's Classifier": BASELINE_PRED_COL,
    'Zero-shot LLM':        'zero_shot_pred',
    'Few-shot LLM':         'few_shot_pred',
    'Chain-of-thought LLM': 'cot_pred',
}

summary_rows = []

for method_name, col in METHODS.items():
    valid = df[df[col] != -1].copy()
    y_true = valid[LABEL_COL]
    y_pred = valid[col]

    macro_f1    = f1_score(y_true, y_pred, average='macro')
    weighted_f1 = f1_score(y_true, y_pred, average='weighted')
    summary_rows.append({
        'Method':       method_name,
        'Macro F1':     round(macro_f1, 4),
        'Weighted F1':  round(weighted_f1, 4),
        'n (valid)':    len(valid),
    })

    print(f'\n=== {method_name} (n={len(valid)}) ===')
    print(classification_report(y_true, y_pred, target_names=['non-toxic', 'toxic']))

summary_df = pd.DataFrame(summary_rows)
print('\n── Summary ──')
print(summary_df.to_string(index=False))

## F1 Bar Chart

In [ ]:
colors = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple']
fig, ax = plt.subplots(figsize=(9, 5))

x      = range(len(summary_df))
bars_m = ax.bar([i - 0.2 for i in x], summary_df['Macro F1'],    width=0.35, label='Macro F1',    color=colors, alpha=0.85)
bars_w = ax.bar([i + 0.2 for i in x], summary_df['Weighted F1'], width=0.35, label='Weighted F1', color=colors, alpha=0.45)

for bar in bars_m:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars_w:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.012,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(list(x))
ax.set_xticklabels(summary_df['Method'], rotation=12, ha='right', fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_ylabel('F1 Score')
ax.set_title("Toxicity Classification F1: Thrisha's Classifier vs LLM Prompting Strategies\n(Davidson dataset)")
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'llm_f1_comparison.png'), dpi=150)
plt.show()
print('Chart saved to results/llm_f1_comparison.png')

## Confusion Matrices (one per method)

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, len(METHODS), figsize=(5 * len(METHODS), 4))

for ax, (method_name, col) in zip(axes, METHODS.items()):
    valid  = df[df[col] != -1]
    cm     = confusion_matrix(valid[LABEL_COL], valid[col])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['non-toxic', 'toxic'],
                yticklabels=['non-toxic', 'toxic'])
    ax.set_title(method_name, fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.suptitle('Confusion Matrices — Davidson Dataset', fontsize=12, y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'llm_confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrices saved to results/llm_confusion_matrices.png')